In [1]:
import touchpy as tp
import numpy as np

In [2]:
comp = tp.Comp('TopChopDatIO.tox', run_mode=tp.CompFlags.INTERNAL_TIME_ASYNC)

In [3]:
def on_layout_change(comp, info):
	print('layout changed:')
	print('in tops:', comp.in_tops.count, comp.in_tops.names)
	print('out tops:', comp.out_tops.count, comp.out_tops.names)
	print('in chops:', comp.in_chops.count, comp.in_chops.names)
	print('out chops:', comp.out_chops.count, comp.out_chops.names)
	print('in dats:', comp.in_dats.count, comp.in_dats.names)
	print('out dats:', comp.out_dats.count, comp.out_dats.names)
	print('pars:', comp.par.count, comp.par.names)
	
comp.set_on_layout_change_callback(on_layout_change, {})

In [4]:
comp.start()

layout changed:
in tops: 3 ['topIn1', 'topIn2', 'topIn3']
out tops: 3 ['topOut1', 'topOut2', 'topOut3']
in chops: 3 ['chopIn1', 'chopIn2', 'chopIn3']
out chops: 3 ['chopOut1', 'chopOut2', 'chopOut3']
in dats: 2 ['datIn1', 'datIn2']
out dats: 2 ['datOut1', 'datOut2']
pars: 14 ['Float', 'Xyzw', 'Translate', 'Menu', 'Rotate', 'Momentary', 'Scale', 'Monitortop', 'File', 'Pulse', 'Int', 'Toggle', 'Rgba', 'Uvw']


In [ ]:
comp.par['Rgba'].set(1.0, 1.0, .2, 1.0)

In [ ]:
comp.par['Rotate'].val = 120.0

In [ ]:
comp.par['Scale'].val = .5

In [ ]:
comp.par['Translate'].set(.1, .1)

In [ ]:
comp.par['Rgba'].set(1.0, 1.0, 1.0, 1.0)
comp.par['Scale'].val = 1.0
comp.par['Translate'].set(0, .0)
comp.par['Rotate'].val = 0.0

In [ ]:
comp.par['Monitortop'].val = 'topOut3'

In [ ]:
arr = comp.out_chops[0].as_numpy()
names = comp.out_chops[0].chan_names()
arr *= 2
comp.in_chops[0].from_numpy(arr, names)

In [ ]:
datOut1 = comp.out_dats['datOut1']
comp.in_dats['datIn1'].from_table(datOut1.as_table())

In [ ]:
# create a DatTable and fill it with a list of data
datTable = tp.DatTable()
testList = [['g', 'b', 'c'], ['g', 'h', 'i'], ['t', 'w', 'a']]
datTable.from_list(testList)

# comp.in_dats['datIn2'].from_table(datTable)
comp.in_dats['datIn2'].from_list(testList)

In [ ]:

cudamem = comp.out_tops[0].cuda_memory()
comp.in_tops[0].copy_cuda_memory(cudamem)

Set callback and pass data to be passed back

In [5]:
def on_frame(comp, user_data):
	comp.in_chops[2].from_numpy(user_data['array'], user_data['names'])
	user_data['array'][0,0] += 1

	comp.start_next_frame()

test_array = np.array([[0],[2],[3],[4]], dtype=np.float32)
test_array_chan_names = ['frame', 'chan2', 'chan3', 'chan4']
user_data = {'array': test_array, 'names': test_array_chan_names}

comp.set_on_frame_callback(on_frame, user_data)

Set callback and operate on local data

In [6]:
local_data = np.array([[0],[20],[30],[40]], dtype=np.float32)
local_data_chan_names = ['frame', 'local2', 'local3', 'local4']

def on_frame_local(comp, user_data):
	comp.in_chops[2].from_numpy(local_data, local_data_chan_names)
	local_data[0,0] += 1
	
	comp.start_next_frame()


comp.set_on_frame_callback(on_frame_local, {})

Clear existing callback and reset

comp.set_on_frame_callback(clear_callback, {}) is only need if you are editing and setting the same callback (but likely edited). It is not needed if setting a new callback that is not the currently set callback

In [7]:
comp.clear_on_frame_callback()

In [8]:
def on_frame2(comp, user_data):
	# trying editing the callback and running the cell again
	comp.in_chops[2].from_numpy(user_data['array'], user_data['names'])
	user_data['array'][0,0] += 1
	user_data['array'][1,0] += 10

	# after running once uncomment the following lines and run again!
	cudamem = comp.out_tops[0].cuda_memory()
	comp.in_tops[0].copy_cuda_memory(cudamem)

	comp.par['Rotate'].val = comp.par['Rotate'].val + 1

	comp.start_next_frame()

test_array = np.array([[0],[1],[3],[4]], dtype=np.float32)
test_array_chan_names = ['frame', 'chan2', 'chan3', 'chan4']
user_data = {'array': test_array, 'names': test_array_chan_names}

comp.set_on_frame_callback(on_frame2, user_data)

In [9]:
comp.stop()

In [10]:
comp.unload() # unloads the whole instance not just the tox,  

In [ ]:
del(comp) # issues with this if not calling unload() first, only in jupyter...